# Data Modelling — Time Series Forecasting of Solar Power Production

## From Exploration to Prediction

In the previous notebooks we **explored** and **understood** our solar power data. Now we take the next step in the data science workflow: **modelling**. Our goal is to build a model that can **forecast tomorrow's solar power production** given historical patterns.

### Learning Objectives
- Understand the concept of **time series forecasting** and why it matters in renewable energy
- Learn how to prepare data for a forecasting model
- Build and evaluate a baseline model using **Facebook Prophet**
- Improve the model iteratively (tuning + external regressors)
- Understand and apply **backtesting** to validate forecast reliability
- Interpret standard evaluation metrics: MAE, RMSE, R², MAPE

### Why Forecasting Matters in Solar Energy

Accurate day-ahead power forecasts are critical for:
- **Grid operators**: balancing supply and demand on the electrical grid
- **Energy traders**: selling tomorrow's production on the energy market
- **Plant operators**: planning maintenance windows and detecting anomalies



## 1. Data Loading and Preparation

We start by loading the same Plant 2 data we explored in the previous notebooks and applying the same quality filters.

Since we already performed a thorough data understanding, we can reuse the data preparation steps directly.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Load datasets
generation_df = pd.read_csv('../data/Plant_2_Generation_Data.csv')
weather_df = pd.read_csv('../data/Plant_2_Weather_Sensor_Data.csv')

# Convert DATE_TIME
generation_df['DATE_TIME'] = pd.to_datetime(generation_df['DATE_TIME'])
weather_df['DATE_TIME'] = pd.to_datetime(weather_df['DATE_TIME'])

# Create DAY and TIME columns
generation_df['DAY'] = generation_df['DATE_TIME'].dt.date
generation_df['DAYTIME'] = generation_df['DATE_TIME'].dt.time

print(f"Generation data: {generation_df.shape}")
print(f"Unique inverters: {generation_df['SOURCE_KEY'].nunique()}")
print(f"Date range: {generation_df['DATE_TIME'].min()} to {generation_df['DATE_TIME'].max()}")

### 1.1 Data Quality Filter

As we discovered in the previous exploration, Plant 2 has two clusters of inverters with different data completeness. We keep only the high-quality cluster (>98% completeness) to avoid noise from missing data corrupting our forecast.

In [ ]:
# Calculate data completeness
total_sample_per_day = 96
source_dataquality = generation_df.groupby(['DAY','SOURCE_KEY'])[['DAYTIME']].count().rename(columns={'DAYTIME': 'sample_completeness'})
source_dataquality['sample_completeness'] = round(source_dataquality['sample_completeness'] / total_sample_per_day * 100, 2)

# Create heatmap data
heatmap_data = source_dataquality.reset_index().pivot(index='SOURCE_KEY', columns='DAY', values='sample_completeness')

# Select SOURCE_KEYs with more than 98% average completeness
completeness_threshold = 98.0
source_avg_completeness = heatmap_data.mean(axis=1)
high_quality_sources = list(source_avg_completeness[source_avg_completeness > completeness_threshold].index)

print(f"High-quality sources: {len(high_quality_sources)} out of {len(source_avg_completeness)}")
print(f"Selected: {high_quality_sources}")

# Filter data
generation_df = generation_df[generation_df.SOURCE_KEY.isin(high_quality_sources)]
print(f"Filtered data shape: {generation_df.shape}")

## 2. Introduction to Time Series Forecasting

### What is a Time Series?

A **time series** is a sequence of data points ordered by time. Our solar power data — DC power measured every 15 minutes — is a classic time series.

### What is Forecasting?

Forecasting means **predicting future values** based on patterns observed in past data. In our case:
- **Input:** Historical power production measurements
- **Output:** Predicted power production for the next day

### Key Concepts We'll Use

| Concept | Meaning |
|---|---|
| **Seasonality** | Repeating patterns at fixed intervals (e.g., daily: power rises at sunrise, peaks at noon, drops at sunset) |
| **Trend** | Long-term increase or decrease in the data |
| **External Regressor** | An additional variable (like weather) that helps improve the forecast |
| **Train/Test Split** | Dividing data into a *training set* (to learn from) and a *test set* (to evaluate on) |
| **Backtesting** | Simulating how the model would have performed on past days — the gold standard for forecast evaluation |

### What is Facebook Prophet?

**Prophet** is an open-source forecasting library developed by Meta (Facebook). It is designed for business time series with the following characteristics:
- Strong **daily/weekly/yearly seasonality**
- **Holidays** or special events
- **Missing data** and outliers

Prophet decomposes a time series into:

$$y(t) = \text{trend}(t) + \text{seasonality}(t) + \text{regressors}(t) + \text{error}(t)$$

> **Why Prophet for this lesson?** It's intuitive, requires minimal configuration, and is designed so that **domain experts** (not just data scientists) can interact with and improve the model.

## 3. Preparing the Time Series

Prophet requires a DataFrame with exactly two columns:
- `ds` — the datetime column
- `y` — the value to forecast

We aggregate DC_POWER across all inverters (summing them) to get the **total plant production** at each 15-minute timestamp.

In [ ]:
# Import Prophet and evaluation metrics
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Aggregate DC_POWER by DATE_TIME (sum across all inverters)
ts_data = generation_df.groupby('DATE_TIME')['DC_POWER'].sum().reset_index()
ts_data.columns = ['ds', 'y']  # Prophet requires 'ds' and 'y' column names

print(f"Time series data shape: {ts_data.shape}")
print(f"Date range: {ts_data['ds'].min()} to {ts_data['ds'].max()}")
print(f"\nFirst few rows:")
print(ts_data.head())

### 3.1 Train/Test Split

To evaluate our forecast, we need to **hold out** data that the model has never seen.

Our strategy: use **all days except the last one** for training, and the **last day** as the test set. This simulates the real-world scenario: "Can we predict tomorrow's production using all historical data up to today?"

> **Important:** In time series, we **never shuffle** the data randomly. The test set must always be **in the future** relative to the training set — otherwise we would be "leaking" future information into the model.

In [ ]:
# Prepare for backtesting: split data
# We use all data except the last day for training, and the last day for testing
unique_days = sorted(ts_data['ds'].dt.date.unique())
test_day = unique_days[-1]
train_days = unique_days[:-1]

print(f"Total days: {len(unique_days)}")
print(f"Training days: {len(train_days)} (from {train_days[0]} to {train_days[-1]})")
print(f"Test day: {test_day}")

# Split the data
train_data = ts_data[ts_data['ds'].dt.date.isin(train_days)]
test_data = ts_data[ts_data['ds'].dt.date == test_day]

print(f"\nTraining data: {len(train_data)} records")
print(f"Test data: {len(test_data)} records")

## 4. Baseline Model — Prophet with Default Settings

Let's start with the simplest possible Prophet model to establish a **baseline**. In data science, it's always good practice to start simple and add complexity only when justified by better results.

We configure:
- `daily_seasonality=True` — solar power has an obvious daily pattern (sunrise → sunset)
- `weekly_seasonality=False` — solar panels don't care what day of the week it is
- `yearly_seasonality=False` — we only have 34 days of data, not enough for yearly patterns
- `seasonality_mode='multiplicative'` — the daily pattern *scales* with the overall level (e.g., more irradiation = larger amplitude)

In [ ]:
# Train the baseline Prophet model
model = Prophet(
    daily_seasonality=True,
    weekly_seasonality=False,
    yearly_seasonality=False,
    seasonality_mode='multiplicative',
)

print("Training Prophet model...")
model.fit(train_data)
print("Model training completed!")

### 4.1 Generate Predictions

We ask Prophet to create a "future" dataframe that extends from the training period into the test day, then call `.predict()` to generate forecasts.

Prophet produces three key columns:
- `yhat` — the point forecast (best guess)
- `yhat_lower` / `yhat_upper` — the **uncertainty interval** (confidence band)

In [ ]:
# Make predictions for the test day (1 day ahead)
future = model.make_future_dataframe(periods=len(test_data), freq='15min')
forecast = model.predict(future)

# Extract predictions for the test day only
test_predictions = forecast[forecast['ds'].dt.date == test_day][['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
test_predictions = test_predictions.merge(test_data, on='ds', how='inner')

print(f"Predictions generated: {len(test_predictions)} records")
print(f"\nSample predictions:")
print(test_predictions.head())

### 4.2 Evaluation Metrics

Before looking at the results, let's understand what each metric tells us:

| Metric | Full Name | What It Measures | Interpretation |
|---|---|---|---|
| **MAE** | Mean Absolute Error | Average size of errors (in kW) | "On average, the forecast is off by X kW" |
| **RMSE** | Root Mean Squared Error | Like MAE but penalizes large errors more | Higher than MAE if there are big spikes in error |
| **R²** | Coefficient of Determination | How much variance the model explains | 1.0 = perfect, 0.0 = no better than guessing the mean |
| **MAPE** | Mean Absolute Percentage Error | Average error as a percentage | "On average, the forecast is off by X%" |

> **Which metric to focus on?**  **MAE** is often preferred because it directly translates to monetary value (€/kW). **MAPE** is useful for communicating results to non-technical stakeholders.

In [ ]:
# Evaluate test performance
mae = mean_absolute_error(test_predictions['y'], test_predictions['yhat'])
rmse = np.sqrt(mean_squared_error(test_predictions['y'], test_predictions['yhat']))
r2 = r2_score(test_predictions['y'], test_predictions['yhat'])

# Calculate MAPE (only for non-zero actual values to avoid division by zero)
non_zero_mask = test_predictions['y'] > 0
mape = np.mean(np.abs((test_predictions.loc[non_zero_mask, 'y'] - test_predictions.loc[non_zero_mask, 'yhat']) / 
                      test_predictions.loc[non_zero_mask, 'y'])) * 100

print("=" * 50)
print("BASELINE MODEL — 1 DAY AHEAD FORECAST")
print("=" * 50)
print(f"Mean Absolute Error (MAE):  {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² Score: {r2:.4f}")
print(f"Mean Abs. Percentage Error (MAPE): {mape:.2f}%")
print("=" * 50)

### 4.3 Visualize the Forecast

Two plots to assess our baseline:

1. **Actual vs Predicted** — do the curves match? Does the confidence interval capture the actual values?
2. **Residuals** (Actual - Predicted) — are errors random or do they show a pattern? Systematic patterns in residuals indicate the model is missing something.

In [ ]:
# Visualize test results
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Plot 1: Actual vs Predicted
axes[0].plot(test_predictions['ds'], test_predictions['y'], 'b-', label='Actual', linewidth=2)
axes[0].plot(test_predictions['ds'], test_predictions['yhat'], 'r--', label='Predicted', linewidth=2)
axes[0].fill_between(test_predictions['ds'], 
                      test_predictions['yhat_lower'], 
                      test_predictions['yhat_upper'], 
                      alpha=0.3, color='red', label='Confidence Interval')
axes[0].set_title(f'Prophet Baseline Forecast — 1 Day Ahead ({test_day})', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Time', fontsize=12)
axes[0].set_ylabel('DC Power (kW)', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Residuals
residuals = test_predictions['y'] - test_predictions['yhat']
axes[1].plot(test_predictions['ds'], residuals, 'g-', linewidth=1.5)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Forecast Residuals (Actual - Predicted)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time', fontsize=12)
axes[1].set_ylabel('Residual (kW)', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean Residual: {residuals.mean():.2f}")
print(f"Std Residual: {residuals.std():.2f}")

### 4.4 Prophet Components — Understanding What the Model Learned

Prophet allows us to decompose the forecast into its building blocks. This is one of its most powerful features: **interpretability**.

Let's look at what the model has learned about the trend and daily seasonality.

In [ ]:
# Prophet component decomposition
fig = model.plot_components(forecast)
plt.tight_layout()
plt.show()

### Think About It

Look at Prophet's component plots carefully:

- Does the **trend** component make physical sense? Is solar power production really changing that much over 34 days?
- Does the **daily seasonality** shape match what you'd expect from a solar plant?

The key feature of Prophet is its ability to engage **human-in-the-loop**. Business people who don't know how to code can observe these plots and suggest improvements based on their domain knowledge.

> **Observation:** The trend is probably too flexible — it's fitting noise rather than capturing a real phenomenon. We can fix this by making the trend more rigid.

## 5. Improved Model — Tuning Prophet

Based on our observations of the baseline model's components, we make two targeted improvements:

1. **More rigid trend** (`changepoint_prior_scale=0.001`): Solar power production shouldn't change dramatically over 34 days — the trend should be nearly flat. Lowering this parameter prevents Prophet from overfitting the trend to noise.

2. **Enhanced daily seasonality** (`fourier_order=15`): Solar power curves have a distinctive shape (asymmetric bell curve). A higher Fourier order gives Prophet more flexibility to capture this shape precisely.

We also disable the default `daily_seasonality` and add our own custom version with these enhanced settings.

> **Key Lesson:** Model improvement often comes from applying **domain knowledge** to constrain or enhance specific components, not from blindly tuning parameters.

In [ ]:
# Train improved Prophet model with more rigid trend and enhanced seasonality
model = Prophet(
    daily_seasonality=False,       # We'll add our own enhanced version
    weekly_seasonality=False,
    yearly_seasonality=False,
    seasonality_mode='multiplicative',
    changepoint_prior_scale=0.001  # Much more rigid trend (default is 0.05)
)

# Add custom daily seasonality with higher Fourier order for more flexibility
model.add_seasonality(
    name='daily_enhanced',
    period=1,           # 1-day period
    fourier_order=15,   # Higher order = more flexible shape (default is 3-5)
    prior_scale=10.0    # Higher scale = let data drive this component
)

print("Training improved Prophet model...")
model.fit(train_data)
print("Model training completed!")

In [ ]:
# Generate predictions with the improved model
future = model.make_future_dataframe(periods=len(test_data), freq='15min')
forecast = model.predict(future)

# Clip negative predictions to 0 (solar power cannot be negative!)
forecast['yhat'] = forecast['yhat'].clip(lower=0)
forecast['yhat_lower'] = forecast['yhat_lower'].clip(lower=0)
forecast['yhat_upper'] = forecast['yhat_upper'].clip(lower=0)

# Extract predictions for the test day
test_predictions = forecast[forecast['ds'].dt.date == test_day][['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
test_predictions = test_predictions.merge(test_data, on='ds', how='inner')

### 5.1 Check the Improved Components

Let's see if the trend is now more stable and the daily pattern more realistic.

In [ ]:
# Plot the improved model's components
fig = model.plot_components(forecast)
plt.tight_layout()
plt.show()

### 5.2 Compare Baseline vs Improved Model

Let's visualize the forecast and compare the metrics side by side.

In [ ]:
# Visualize improved model results
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Plot 1: Actual vs Predicted
axes[0].plot(test_predictions['ds'], test_predictions['y'], 'b-', label='Actual', linewidth=2)
axes[0].plot(test_predictions['ds'], test_predictions['yhat'], 'r--', label='Predicted', linewidth=2)
axes[0].fill_between(test_predictions['ds'], 
                      test_predictions['yhat_lower'], 
                      test_predictions['yhat_upper'], 
                      alpha=0.3, color='red', label='Confidence Interval')
axes[0].set_title(f'Improved Prophet Forecast — 1 Day Ahead ({test_day})', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Time', fontsize=12)
axes[0].set_ylabel('DC Power (kW)', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Residuals
residuals = test_predictions['y'] - test_predictions['yhat']
axes[1].plot(test_predictions['ds'], residuals, 'g-', linewidth=1.5)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Forecast Residuals', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time', fontsize=12)
axes[1].set_ylabel('Residual (kW)', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean Residual: {residuals.mean():.2f}")
print(f"Std Residual: {residuals.std():.2f}")

In [ ]:
# Compare baseline vs improved metrics
print("=" * 50)
print("BASELINE MODEL RESULTS")
print("=" * 50)
print(f"  MAE: {mae:.2f} | RMSE: {rmse:.2f} | R²: {r2:.4f} | MAPE: {mape:.2f}%")
print(" " * 50)

# Compute improved model metrics
mae_improved = mean_absolute_error(test_predictions['y'], test_predictions['yhat'])
rmse_improved = np.sqrt(mean_squared_error(test_predictions['y'], test_predictions['yhat']))
r2_improved = r2_score(test_predictions['y'], test_predictions['yhat'])
non_zero_mask = test_predictions['y'] > 0
mape_improved = np.mean(np.abs((test_predictions.loc[non_zero_mask, 'y'] - test_predictions.loc[non_zero_mask, 'yhat']) / 
                       test_predictions.loc[non_zero_mask, 'y'])) * 100

print("=" * 50)
print("IMPROVED MODEL RESULTS")
print("=" * 50)
print(f"  MAE: {mae_improved:.2f} | RMSE: {rmse_improved:.2f} | R²: {r2_improved:.4f} | MAPE: {mape_improved:.2f}%")
print("=" * 50)

# Update metrics for later comparison
mae = mae_improved
rmse = rmse_improved
r2 = r2_improved
mape = mape_improved

## 6. Adding Weather Data — External Regressors

So far, our model only knows about historical power production patterns. But we learned from the exploration notebooks that **irradiation** is a strong driver of power generation.

Prophet supports **external regressors** — additional variables that the model can use alongside its time-based components. We'll add IRRADIATION as a regressor to see if it improves the forecast.

> **Real-world note:** In production, the irradiation for *tomorrow* wouldn't be known exactly — you'd use a **weather forecast**. For this lesson, we use actual measured values to isolate the effect of the regressor from weather forecast errors.

### 6.1 Merge Weather Data

In [ ]:
# Merge weather data with time series data
print("Merging weather data with generation data...")

# Aggregate weather data by DATE_TIME (mean across sensors)
weather_agg = weather_df.groupby('DATE_TIME').agg({
    'IRRADIATION': 'mean',
    'AMBIENT_TEMPERATURE': 'mean',
    'MODULE_TEMPERATURE': 'mean'
}).reset_index()

# Merge with time series data
ts_data_with_weather = ts_data.merge(weather_agg, left_on='ds', right_on='DATE_TIME', how='inner')
ts_data_with_weather = ts_data_with_weather.drop('DATE_TIME', axis=1)

print(f"Merged data shape: {ts_data_with_weather.shape}")
print(f"\nMerged data columns: {list(ts_data_with_weather.columns)}")

In [ ]:
# Split the weather-enhanced data using the same train/test days
train_data = ts_data_with_weather[ts_data_with_weather['ds'].dt.date.isin(train_days)]
test_data = ts_data_with_weather[ts_data_with_weather['ds'].dt.date == test_day]

print(f"Training data with weather: {len(train_data)} records")
print(f"Test data with weather: {len(test_data)} records")

### 6.2 Train the Weather-Enhanced Model

We use the same tuned configuration as before, but now add IRRADIATION as an external regressor using `add_regressor()`.

In [ ]:
# Train Prophet model with IRRADIATION as external regressor
model_with_weather = Prophet(
    daily_seasonality=False,
    weekly_seasonality=False,
    yearly_seasonality=False,
    seasonality_mode='multiplicative',
    changepoint_prior_scale=0.001
)

# Add enhanced daily seasonality (same as before)
model_with_weather.add_seasonality(
    name='daily_enhanced',
    period=1,
    fourier_order=15,
    prior_scale=10.0
)

# Add IRRADIATION as external regressor
model_with_weather.add_regressor('IRRADIATION', prior_scale=0.05)

print("Training Prophet model with IRRADIATION regressor...")
model_with_weather.fit(train_data)
print("Model training completed!")

### 6.3 Generate Weather-Enhanced Predictions

When using external regressors, we must provide the regressor values for the future period too. Prophet's `make_future_dataframe()` creates the time index, but we need to manually add the IRRADIATION column.

In [ ]:
# Make predictions with weather data
future_weather = model_with_weather.make_future_dataframe(periods=len(test_data), freq='15min')

# Add IRRADIATION values to future dataframe
train_irradiation = train_data[['ds', 'IRRADIATION']].set_index('ds')
test_irradiation = test_data[['ds', 'IRRADIATION']].set_index('ds')

# Merge irradiation data
future_weather = future_weather.set_index('ds')
future_weather['IRRADIATION'] = pd.concat([train_irradiation, test_irradiation])['IRRADIATION']
future_weather = future_weather.reset_index()

print(f"Future dataframe with IRRADIATION: {future_weather.shape}")
print(f"Missing IRRADIATION values: {future_weather['IRRADIATION'].isnull().sum()}")

# Make forecast
forecast_weather = model_with_weather.predict(future_weather)

# Clip negative predictions to 0
forecast_weather['yhat'] = forecast_weather['yhat'].clip(lower=0)
forecast_weather['yhat_lower'] = forecast_weather['yhat_lower'].clip(lower=0)
forecast_weather['yhat_upper'] = forecast_weather['yhat_upper'].clip(lower=0)

# Extract predictions for the test day
test_predictions_weather = forecast_weather[forecast_weather['ds'].dt.date == test_day][['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
test_predictions_weather = test_predictions_weather.merge(test_data, on='ds', how='inner')

print(f"Weather-enhanced predictions generated: {len(test_predictions_weather)} records")
print(f"\nSample predictions:")
print(test_predictions_weather[['ds', 'y', 'yhat', 'IRRADIATION']].head())

### 6.4 Model Comparison — All Three Versions

Let's compare all three models side by side to see the progression of improvement.

In [ ]:
# Evaluate weather-enhanced model performance
mae_weather = mean_absolute_error(test_predictions_weather['y'], test_predictions_weather['yhat'])
rmse_weather = np.sqrt(mean_squared_error(test_predictions_weather['y'], test_predictions_weather['yhat']))
r2_weather = r2_score(test_predictions_weather['y'], test_predictions_weather['yhat'])

non_zero_mask_weather = test_predictions_weather['y'] > 0
mape_weather = np.mean(np.abs((test_predictions_weather.loc[non_zero_mask_weather, 'y'] - 
                               test_predictions_weather.loc[non_zero_mask_weather, 'yhat']) / 
                              test_predictions_weather.loc[non_zero_mask_weather, 'y'])) * 100

print("=" * 60)
print("MODEL COMPARISON — 1 DAY AHEAD FORECAST")
print("=" * 60)
print("TUNED MODEL (no weather):")
print(f"  MAE: {mae:.2f} | RMSE: {rmse:.2f} | R²: {r2:.4f} | MAPE: {mape:.2f}%")
print("")
print("TUNED MODEL + IRRADIATION REGRESSOR:")
print(f"  MAE: {mae_weather:.2f} | RMSE: {rmse_weather:.2f} | R²: {r2_weather:.4f} | MAPE: {mape_weather:.2f}%")
print("")
print("IMPROVEMENT from adding weather:")
print(f"  MAE: {((mae - mae_weather) / mae * 100):+.1f}%")
print(f"  RMSE: {((rmse - rmse_weather) / rmse * 100):+.1f}%")
print(f"  R²: {((r2_weather - r2) / abs(r2) * 100):+.1f}%")
print(f"  MAPE: {((mape - mape_weather) / mape * 100):+.1f}%")
print("=" * 60)

In [ ]:
# Visual comparison: all models
fig, axes = plt.subplots(3, 1, figsize=(18, 15))

# Plot 1: Actual vs Predicted (both models)
axes[0].plot(test_predictions['ds'], test_predictions['y'], 'b-', label='Actual', linewidth=2)
axes[0].plot(test_predictions['ds'], test_predictions['yhat'], 'r--', label='Predicted (No Weather)', linewidth=2, alpha=0.7)
axes[0].plot(test_predictions_weather['ds'], test_predictions_weather['yhat'], 'g--', label='Predicted (With IRRADIATION)', linewidth=2)
axes[0].fill_between(test_predictions_weather['ds'], 
                      test_predictions_weather['yhat_lower'], 
                      test_predictions_weather['yhat_upper'], 
                      alpha=0.2, color='green', label='Confidence Interval (Weather)')
axes[0].set_title(f'Prophet Forecast Comparison — 1 Day Ahead ({test_day})', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Time', fontsize=12)
axes[0].set_ylabel('DC Power (kW)', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Residuals comparison
residuals_no_weather = test_predictions['y'] - test_predictions['yhat']
residuals_with_weather = test_predictions_weather['y'] - test_predictions_weather['yhat']

axes[1].plot(test_predictions['ds'], residuals_no_weather, 'r-', linewidth=1.5, alpha=0.7, label='No Weather')
axes[1].plot(test_predictions_weather['ds'], residuals_with_weather, 'g-', linewidth=1.5, label='With IRRADIATION')
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Forecast Residuals Comparison', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time', fontsize=12)
axes[1].set_ylabel('Residual (kW)', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Plot 3: IRRADIATION values for context
axes[2].plot(test_data['ds'], test_data['IRRADIATION'], 'orange', linewidth=2)
axes[2].set_title('Irradiation Values for the Test Day', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Time', fontsize=12)
axes[2].set_ylabel('Irradiation (kW/m²)', fontsize=12)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nResiduals Statistics Comparison:")
print(f"No Weather    — Mean: {residuals_no_weather.mean():.2f}, Std: {residuals_no_weather.std():.2f}")
print(f"With Weather  — Mean: {residuals_with_weather.mean():.2f}, Std: {residuals_with_weather.std():.2f}")

In [ ]:
# Weather-enhanced model components
fig = model_with_weather.plot_components(forecast_weather)
plt.tight_layout()
plt.show()

print("Weather-Enhanced Prophet Components:")
print("  - Trend: Overall time trend")
print("  - daily_enhanced: Enhanced daily seasonality pattern")
print("  - extra_regressors_additive: IRRADIATION regressor contribution")

## 7. Rolling Backtest — Validating on Multiple Days

### Why One Test Day Is Not Enough

Testing on a single day is like grading a student on one question: the result could be lucky or unlucky. To truly trust our model, we need to test it on **multiple days**.

### What is Backtesting?

**Backtesting** (or rolling validation) simulates what would have happened if we had used our model in production over the past N days:

1. For each test day, train on **all data before** that day
2. Forecast the test day
3. Compare the forecast to what actually happened
4. Move to the next day and repeat

This gives us a distribution of performance metrics instead of a single number, allowing us to assess both the **average quality** and the **consistency** of the forecast.

```
Day 1..27 → Train → Forecast Day 28 → Evaluate
Day 1..28 → Train → Forecast Day 29 → Evaluate
Day 1..29 → Train → Forecast Day 30 → Evaluate
...
```



In [ ]:
def backtest_prophet_rolling(ts_data, n_test_days=7):
    """
    Perform rolling backtest for Prophet model with IRRADIATION regressor.
    
    For each of the last n_test_days, trains on all previous data and 
    forecasts 1 day ahead.
    
    Parameters:
    -----------
    ts_data : pd.DataFrame
        Time series data with 'ds', 'y', and 'IRRADIATION' columns
    n_test_days : int
        Number of days to use for backtesting
    
    Returns:
    --------
    results : dict with 'predictions', 'daily_metrics', 'test_days'
    """
    unique_days = sorted(ts_data['ds'].dt.date.unique())
    test_days = unique_days[-n_test_days:]
    
    print(f"Starting rolling backtest for {n_test_days} days...")
    print(f"Test period: {test_days[0]} to {test_days[-1]}")
    print("=" * 60)
    
    all_predictions = []
    daily_metrics = []
    
    for i, test_day in enumerate(test_days, 1):
        print(f"\n[Day {i}/{n_test_days}] Testing: {test_day}")
        
        # Split: all days before test_day for training
        train_data = ts_data[ts_data['ds'].dt.date < test_day]
        test_data = ts_data[ts_data['ds'].dt.date == test_day]
        
        if len(test_data) == 0:
            print(f"  No data available for {test_day}, skipping...")
            continue
        
        print(f"  Training samples: {len(train_data)} | Test samples: {len(test_data)}")
        
        # Train Prophet with same configuration as our best model
        model = Prophet(
            daily_seasonality=False,
            weekly_seasonality=False,
            yearly_seasonality=False,
            seasonality_mode='multiplicative',
            changepoint_prior_scale=0.001
        )
        model.add_seasonality(name='daily_enhanced', period=1, fourier_order=15, prior_scale=10.0)
        model.add_regressor('IRRADIATION', prior_scale=0.8)
        model.fit(train_data)

        # Prepare future with IRRADIATION data
        future = model.make_future_dataframe(periods=len(test_data), freq='15min')
        train_irr = train_data[['ds', 'IRRADIATION']].set_index('ds')
        test_irr = test_data[['ds', 'IRRADIATION']].set_index('ds')
        future = future.set_index('ds')
        future['IRRADIATION'] = pd.concat([train_irr, test_irr])['IRRADIATION']
        future = future.reset_index()
        
        # Forecast and clip negatives
        forecast = model.predict(future)
        forecast['yhat'] = forecast['yhat'].clip(lower=0)
        forecast['yhat_lower'] = forecast['yhat_lower'].clip(lower=0)
        forecast['yhat_upper'] = forecast['yhat_upper'].clip(lower=0)
        
        # Extract test day predictions
        day_preds = forecast[forecast['ds'].dt.date == test_day][['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
        day_preds = day_preds.merge(test_data, on='ds', how='inner')
        
        # Calculate metrics
        day_mae = mean_absolute_error(day_preds['y'], day_preds['yhat'])
        day_rmse = np.sqrt(mean_squared_error(day_preds['y'], day_preds['yhat']))
        day_r2 = r2_score(day_preds['y'], day_preds['yhat'])
        
        non_zero = day_preds['y'] > 0
        day_mape = np.mean(np.abs((day_preds.loc[non_zero, 'y'] - day_preds.loc[non_zero, 'yhat']) / 
                                   day_preds.loc[non_zero, 'y'])) * 100 if non_zero.sum() > 0 else np.nan
        
        print(f"  MAE: {day_mae:.2f} | RMSE: {day_rmse:.2f} | R²: {day_r2:.4f} | MAPE: {day_mape:.2f}%")
        
        all_predictions.append(day_preds)
        daily_metrics.append({
            'test_day': test_day, 'mae': day_mae, 'rmse': day_rmse, 
            'r2': day_r2, 'mape': day_mape, 'n_samples': len(day_preds)
        })
    
    print("\n" + "=" * 60)
    print("Backtesting completed!")
    print("=" * 60)
    
    return {
        'predictions': pd.concat(all_predictions, ignore_index=True),
        'daily_metrics': pd.DataFrame(daily_metrics),
        'test_days': test_days
    }

# Run the rolling backtest on the last 7 days
backtest_results = backtest_prophet_rolling(ts_data_with_weather, n_test_days=7)

### 7.1 Backtest Summary

Let's look at the overall performance across all 7 test days. The **standard deviation** (±) tells us how consistent the model is — lower is better.

In [ ]:
# Display overall backtest metrics
daily_metrics = backtest_results['daily_metrics']

print("=" * 60)
print("OVERALL BACKTEST METRICS (7-DAY ROLLING FORECAST)")
print("=" * 60)
print(f"Average MAE:  {daily_metrics['mae'].mean():.2f} +/- {daily_metrics['mae'].std():.2f}")
print(f"Average RMSE: {daily_metrics['rmse'].mean():.2f} +/- {daily_metrics['rmse'].std():.2f}")
print(f"Average R²:   {daily_metrics['r2'].mean():.4f} +/- {daily_metrics['r2'].std():.4f}")
print(f"Average MAPE: {daily_metrics['mape'].mean():.2f}% +/- {daily_metrics['mape'].std():.2f}%")
print("=" * 60)

print("\nDaily Performance Breakdown:")
print(daily_metrics.to_string(index=False))

### 7.2 Visualize Backtest Performance

Let's see how the error varies across test days. Days with unusual weather (clouds, rain) will naturally have higher error.

In [ ]:
# Visualize backtest performance over time
fig, axes = plt.subplots(2, 1, figsize=(15, 12))

# Plot 1: MAE over test days
axes[0].plot(daily_metrics['test_day'], daily_metrics['mae'], 'o-', label='MAE', linewidth=2, markersize=8)
axes[0].axhline(y=daily_metrics['mae'].mean(), color='red', linestyle='--', alpha=0.5, label=f"Average MAE = {daily_metrics['mae'].mean():.0f}")
axes[0].set_title('Mean Absolute Error by Test Day', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Test Day', fontsize=12)
axes[0].set_ylabel('MAE (kW)', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: MAPE over test days
axes[1].plot(daily_metrics['test_day'], daily_metrics['mape'], '^-', color='orange', 
             label='MAPE', linewidth=2, markersize=8)
axes[1].axhline(y=daily_metrics['mape'].mean(), color='red', linestyle='--', alpha=0.5, label=f"Average MAPE = {daily_metrics['mape'].mean():.1f}%")
axes[1].set_title('Mean Absolute Percentage Error by Test Day', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Test Day', fontsize=12)
axes[1].set_ylabel('MAPE (%)', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 7.3 Full Backtest: Predictions vs Actuals

Let's overlay all 7 days of predictions against actual values to get a full picture of model performance.

In [ ]:
# Visualize predictions vs actuals for all test days
predictions_df = backtest_results['predictions']

fig, axes = plt.subplots(2, 1, figsize=(18, 12))

# Plot 1: All predictions vs actuals
axes[0].plot(predictions_df['ds'], predictions_df['y'], 'b-', label='Actual', linewidth=1.5, alpha=0.7)
axes[0].plot(predictions_df['ds'], predictions_df['yhat'], 'r--', label='Predicted', linewidth=1.5, alpha=0.7)
axes[0].fill_between(predictions_df['ds'], 
                      predictions_df['yhat_lower'], 
                      predictions_df['yhat_upper'], 
                      alpha=0.2, color='red', label='Confidence Interval')
axes[0].set_title('7-Day Rolling Backtest: Actual vs Predicted DC Power', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date Time', fontsize=12)
axes[0].set_ylabel('DC Power (kW)', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot 2: Residuals over time
residuals = predictions_df['y'] - predictions_df['yhat']
axes[1].plot(predictions_df['ds'], residuals, 'g-', linewidth=1, alpha=0.7)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1.5)
axes[1].fill_between(predictions_df['ds'], residuals, 0, alpha=0.3, color='green')
axes[1].set_title('Forecast Residuals Over 7 Test Days', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Date Time', fontsize=12)
axes[1].set_ylabel('Residual (kW)', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nOverall Residuals Statistics:")
print(f"Mean Residual: {residuals.mean():.2f}")
print(f"Std Residual: {residuals.std():.2f}")
print(f"Min Residual: {residuals.min():.2f}")
print(f"Max Residual: {residuals.max():.2f}")

## 8. Summary and Key Takeaways

### What We Accomplished

We built a **day-ahead solar power forecast** through an iterative process:

| Step | Model | Key Change |
|---|---|---|
| Baseline | Prophet default | daily seasonality only |
| Tuned | Rigid trend + enhanced seasonality | Applied domain knowledge to constrain the model |
| With Weather | Added IRRADIATION regressor | Included external data source |
| Backtest | Rolling 7-day validation | Validated reliability across multiple days |

### Key Lessons

1. **Start simple, add complexity only when justified.** The baseline told us what to fix.
2. **Domain knowledge beats blind tuning.** Knowing that solar power shouldn't have a strong 34-day trend guided us to make the trend rigid.
3. **External data (weather) improves forecasts** — but in production you'd need a weather forecast, which itself has errors.
4. **Always backtest.** A single train/test split can be misleading. Rolling backtesting gives you a realistic picture of model performance.
5. **Interpretability matters.** Prophet's component plots let non-technical stakeholders understand and trust the model.

